# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-09 — Validation and Research Claim Audit

**Lane:** Refresh / Content Opportunity Scoring  
**Primary metric:** Precision@50  
**Purpose:** Human decision-support for content-review prioritisation

This notebook audits the Week-5 model using an honest validation design,
a leakage review, real failure examples, and public-safe claim language.

## 1. Two paper findings + my methodology questions

### Finding 1: Growing content was longer and younger than declining content

The paper reports that content with rising impressions was, on average,
longer and younger than content with falling impressions. Growing pages
averaged approximately 3,180 words and 184 days of age, while declining
pages averaged approximately 2,311 words and 230 days of age.

### Methodology questions

**1. How was the growth or decline label created?**

The paper explains that trend direction is based on the change in
impressions between the latest 30-day period and the previous 30-day
period. I would ask whether pages with low impression counts were
filtered or stabilised before assigning the label, because small
absolute changes can create large percentage movements.

**2. Does the validation design support the claim?**

The comparison is an observational cohort comparison rather than a
predictive validation experiment. The sample supports the measured
difference within the observed portfolio, but it does not establish
that increasing word count or reducing content age will cause growth.

Client, topic, search demand, publication timing, and existing
visibility may explain part of the observed difference.

A public-safe interpretation is that longer and younger pages were
associated with stronger recent impression trends in the observed
portfolio. The result is directional and may support review
prioritisation, but it is not a causal rule.

### Finding 2: Recently refreshed mature content showed stronger measured performance

The paper reports that the 31–90 day freshness window had the strongest
stable growth-to-decline ratio. It also reports that mature content
refreshed within 30 days had higher measured health and impressions than
older content that had not been refreshed recently.

### Methodology questions

**1. Where does the refresh label come from?**

Freshness is defined as the number of days since the last content
update. I would ask what type of edit qualifies as an update. A full
rewrite, a metadata change, and an automated timestamp update may create
the same freshness value even though they represent different
interventions.

**2. Does the validation design support the claim?**

The comparison is observational. Pages selected for refresh may already
have had stronger historical visibility, greater business value, better
editorial quality, or more search demand.

This creates possible selection bias because refreshed and untouched
pages may not be directly comparable.

A stronger validation design would compare refreshed and unrefreshed
pages with similar prior impressions, position, age, topic, and client
context. A time-aware before-and-after analysis could also test whether
the measured change occurred after the refresh.

A public-safe interpretation is that recently refreshed mature pages
were associated with stronger measured performance in the observed
portfolio. This is a directional decision-support signal rather than
proof that refreshing any page will create the same result.

In [2]:
import pandas as pd

paper_findings = pd.DataFrame(
    [
        {
            "finding": "Growing content was longer and younger",
            "reported_measure_1": "3,180 vs 2,311 average words",
            "reported_measure_2": "184 vs 230 average age in days",
            "label_source": (
                "Latest 30-day impression trend compared with "
                "the previous 30-day period"
            ),
            "evidence_type": "Observational cohort comparison",
            "safe_interpretation": (
                "Longer and younger pages were associated with "
                "stronger recent impression trends in the observed portfolio."
            ),
        },
        {
            "finding": (
                "Recently refreshed mature content showed "
                "stronger measured performance"
            ),
            "reported_measure_1": "3.2x health comparison",
            "reported_measure_2": "57x impression comparison",
            "label_source": "Days since the last recorded content update",
            "evidence_type": "Observational freshness comparison",
            "safe_interpretation": (
                "Recent refresh activity was associated with stronger "
                "measured performance among mature pages."
            ),
        },
    ]
)

display(paper_findings)

assert len(paper_findings) == 2

assert paper_findings[
    "safe_interpretation"
].str.contains(
    "associated",
    case=False,
).all()

assert paper_findings[
    "evidence_type"
].str.contains(
    "observational",
    case=False,
).all()

print(
    "Two research findings and constructive "
    "methodology questions documented."
)

,finding,reported_measure_1,reported_measure_2,label_source,evidence_type,safe_interpretation
0,Growing content was longer and younger,"3,180 vs 2,311 average words",184 vs 230 average age in days,Latest 30-day impression trend compared with t...,Observational cohort comparison,Longer and younger pages were associated with ...
1,Recently refreshed mature content showed stron...,3.2x health comparison,57x impression comparison,Days since the last recorded content update,Observational freshness comparison,Recent refresh activity was associated with st...


Two research findings and constructive methodology questions documented.


## 2. My model under an honest split (before/after)

I re-run the Week-5 Logistic Regression under two validation designs.

### Before — random row split

A random row split can place pages from the same client in both training
and validation. Pages from the same client may share site structure,
measurement patterns, and editorial practices. This may make the
measured validation result optimistic.

### After — grouped client split

The grouped split places each anonymised client entirely in either
training or validation. The same client cannot occur in both sets.

This better represents the deployment question:

> Can the model rank pages belonging to a client it did not observe
> during training?

Both experiments use the same operational population, feature set,
target, Logistic Regression pipeline, test size, random seed, and
Precision@50 calculation.

The grouped result is treated as the more honest result. The random
result is retained only for the required before-and-after comparison.

In [3]:
# Run the completed Week-5 notebook first.
# Both notebooks must be in work/notebooks/.

%run ./w05_model.ipynb

from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from pathlib import Path

required_objects = [
    "model_frame",
    "feature_columns",
    "target_column",
    "group_column",
    "preprocessor",
    "SEED",
    "TEST_SIZE",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Week-5 notebook did not create: "
        + ", ".join(missing_objects)
    )

TOP_K_AUDIT = 50


def build_audit_model():
    return Pipeline(
        steps=[
            (
                "preprocessor",
                clone(preprocessor),
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=SEED,
                ),
            ),
        ]
    )


def precision_at_k_audit(labels, scores, k=50):
    labels_array = np.asarray(
        labels,
        dtype=int,
    )

    scores_array = np.asarray(
        scores,
        dtype=float,
    )

    if len(labels_array) != len(scores_array):
        raise ValueError(
            "Labels and scores must have equal length."
        )

    if len(labels_array) == 0:
        raise ValueError(
            "Evaluation arrays cannot be empty."
        )

    if not np.isfinite(scores_array).all():
        raise ValueError(
            "Scores contain NaN or infinite values."
        )

    effective_k = min(
        k,
        len(labels_array),
    )

    ranked_indices = np.argsort(
        -scores_array,
        kind="mergesort",
    )[:effective_k]

    return float(
        labels_array[ranked_indices].mean()
    )


def evaluate_split(
    split_name,
    validation_frame,
    scores,
    training_rows,
    client_overlap,
):
    labels = validation_frame[
        target_column
    ].to_numpy()

    return {
        "validation_design": split_name,
        "train_rows": int(training_rows),
        "validation_rows": int(
            len(validation_frame)
        ),
        "positive_base_rate": float(
            labels.mean()
        ),
        "client_overlap": int(
            client_overlap
        ),
        "precision@50": precision_at_k_audit(
            labels,
            scores,
            TOP_K_AUDIT,
        ),
        "average_precision": float(
            average_precision_score(
                labels,
                scores,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                labels,
                scores,
            )
        ),
    }


# ---------------------------------------------------------
# BEFORE — random row split
# ---------------------------------------------------------

all_indices = np.arange(
    len(model_frame)
)

random_train_idx, random_validation_idx = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=model_frame[target_column],
)

random_train = model_frame.iloc[
    random_train_idx
].copy()

random_validation = model_frame.iloc[
    random_validation_idx
].copy()

random_train_clients = set(
    random_train[group_column].unique()
)

random_validation_clients = set(
    random_validation[group_column].unique()
)

random_client_overlap = len(
    random_train_clients.intersection(
        random_validation_clients
    )
)

random_model = build_audit_model()

random_model.fit(
    random_train[feature_columns],
    random_train[target_column],
)

random_scores = random_model.predict_proba(
    random_validation[feature_columns]
)[:, 1]

random_result = evaluate_split(
    split_name="Before — random row split",
    validation_frame=random_validation,
    scores=random_scores,
    training_rows=len(random_train),
    client_overlap=random_client_overlap,
)


# ---------------------------------------------------------
# AFTER — grouped client split
# ---------------------------------------------------------

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=SEED,
)

group_train_idx, group_validation_idx = next(
    group_splitter.split(
        model_frame[feature_columns],
        model_frame[target_column],
        groups=model_frame[group_column],
    )
)

group_train = model_frame.iloc[
    group_train_idx
].copy()

group_validation = model_frame.iloc[
    group_validation_idx
].copy()

group_train_clients = set(
    group_train[group_column].unique()
)

group_validation_clients = set(
    group_validation[group_column].unique()
)

group_client_overlap = len(
    group_train_clients.intersection(
        group_validation_clients
    )
)

assert group_client_overlap == 0

grouped_model = build_audit_model()

grouped_model.fit(
    group_train[feature_columns],
    group_train[target_column],
)

grouped_scores = grouped_model.predict_proba(
    group_validation[feature_columns]
)[:, 1]

grouped_result = evaluate_split(
    split_name="After — grouped client split",
    validation_frame=group_validation,
    scores=grouped_scores,
    training_rows=len(group_train),
    client_overlap=group_client_overlap,
)


# ---------------------------------------------------------
# Honest comparison
# ---------------------------------------------------------

split_comparison = pd.DataFrame(
    [
        random_result,
        grouped_result,
    ]
)

display(
    split_comparison.round(4)
)

random_precision_50 = float(
    random_result["precision@50"]
)

grouped_precision_50 = float(
    grouped_result["precision@50"]
)

precision_gap = (
    grouped_precision_50
    - random_precision_50
)

print(
    "Random split Precision@50:",
    round(random_precision_50, 3),
)

print(
    "Grouped split Precision@50:",
    round(grouped_precision_50, 3),
)

print(
    "Grouped minus random difference:",
    round(precision_gap, 3),
)

print(
    "Random split client overlap:",
    random_client_overlap,
)

print(
    "Grouped split client overlap:",
    group_client_overlap,
)

if precision_gap < 0:
    print(
        "Observed result: the grouped estimate is lower. "
        "This is directionally consistent with the random "
        "split being more optimistic."
    )

elif np.isclose(
    precision_gap,
    0,
):
    print(
        "Observed result: the two estimates are similar. "
        "The grouped design remains safer because client "
        "overlap is zero."
    )

else:
    print(
        "Observed result: the grouped estimate is higher "
        "on this holdout. The grouped design remains the "
        "honest design because client overlap is zero."
    )

ML09_OUTPUT_DIR = Path(
    "work/outputs/ml09"
)

ML09_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

split_comparison.to_csv(
    ML09_OUTPUT_DIR
    / "before_after_split_comparison.csv",
    index=False,
)

Exception: File `'./w05_model.ipynb.py'` not found.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.